In [4]:
from langchain_core.documents import Document
from langchain_community.document_loaders.pdf import PyPDFLoader
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

C:\Users\Tanvir\AppData\Local\Temp\ipykernel_23504\3404023348.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader
c:\Users\Tanvir\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### PDF Reader

In [11]:
def load_all_pdf():
    path = "pdf"
    num_pdf= 0
    all_doc = []

    folder_path = os.listdir(path)

    for filename in folder_path:
        if filename.lower().endswith('.pdf'):
            file_path = os.path.join(path, filename)
            loader = PyPDFLoader(file_path)
            doc = loader.load()

            all_doc.extend(doc)
            num_pdf += 1
            print(filename)

    print(f'number of pdf {num_pdf}, pages {len(all_doc)}')
    return all_doc

In [12]:
all_pdf = load_all_pdf()


Attention is all you need.pdf
prompt Engineering.pdf
number of pdf 2, pages 83


### Chunks

In [13]:
def split_chinks(documents, chunk_size = 500, chunk_overlap = 50):
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunked_docs = text_spliter.split_documents(documents)
    return chunked_docs

In [14]:
chunks = split_chinks(all_pdf)
len(chunks)

305

### Embedding

In [15]:
class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading Model...", self.model_name)
        self.model = SentenceTransformer(model_name)
        print("Embedding dimantions", self.model.get_embedding_dimension())

    def generate_embedding(self, text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        print("Embedding Status", embeddings.shape)
        return embeddings
    

In [16]:
embedding_manager = EmbeddingManager()

Loading Model... all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8038.65it/s]


Embedding dimantions 384


### Vector DB

In [17]:
import chromadb
import uuid

In [25]:
class VectorStoreManager:
    def __init__(self, presist_directory="vector_store", collection_name = "pdf_documents"):
        self.presist_directory = presist_directory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.presist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.presist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embedding in RAG"}
        )

        print('initialized the vector store with collection:', self.collection_name)
        print('docs in collection:', self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError('Embedding and Document size not matched!')
        
        ids = []
        all_metadata = []
        documents_content = []
        embedding_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embedding_list.append(emb.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embedding_list
        )

        print('total documents added in vector store:', len(documents_content))
        print('docs in collection:', self.collection.count())

In [26]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [27]:
chunks[0
       ]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdf\\Attention is all you need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu')

In [29]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embedding(texts)

vector_store.add_documents(chunks, embedding)

Batches: 100%|██████████| 10/10 [00:00<00:00, 17.48it/s]


Embedding Status (305, 384)
total documents added in vector store: 305
docs in collection: 305
